# 02d: Preprocessing Validation

**Purpose:** Validate preprocessing pipeline and splits

**Dataset:** COMPAS (transformed and split)

**Date:** 2025-11-08

---

## Overview

### Purpose
- Verify no data leakage
- Check distribution preservation
- Validate transformations
- Generate preprocessing report

### Runtime: <1 minute

---

In [1]:
# Setup
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

project_root = Path.cwd().parent.parent
PROCESSED_DIR = project_root / "data" / "processed"
METADATA_DIR = project_root / "data" / "metadata"
FIGURES_DIR = project_root / "results" / "figures" / "exploratory"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Setup complete")

✓ Setup complete


## 1. Load Splits

In [2]:
# Load train/test
X_train = pd.read_parquet(PROCESSED_DIR / "compas_X_train.parquet")
X_test = pd.read_parquet(PROCESSED_DIR / "compas_X_test.parquet")
y_train = pd.read_parquet(PROCESSED_DIR / "compas_y_train.parquet")['two_year_recid']
y_test = pd.read_parquet(PROCESSED_DIR / "compas_y_test.parquet")['two_year_recid']

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

Train: X=(4167, 9), y=(4167,)
Test:  X=(1042, 9), y=(1042,)


## 2. Check for Data Leakage

Verify no overlap between train and test indices.

In [3]:
# Check index overlap
train_idx = set(X_train.index)
test_idx = set(X_test.index)
overlap = train_idx.intersection(test_idx)

if len(overlap) == 0:
    print("✓ No data leakage: train and test indices are disjoint")
else:
    print(f"⚠ WARNING: {len(overlap)} overlapping indices!")

# Verify total count
total_samples = len(train_idx) + len(test_idx)
print(f"\nTotal unique samples: {total_samples:,}")

⚠ WARNING: 1042 overlapping indices!

Total unique samples: 5,209


## 3. Distribution Preservation

### 3.1 Class Distribution

In [ ]:
# Compare class distributions
print("Class distribution:")
print(f"Train: {y_train.value_counts().to_dict()} ({y_train.mean():.1%} positive)")
print(f"Test:  {y_test.value_counts().to_dict()} ({y_test.mean():.1%} positive)")

# Chi-squared test
train_counts = y_train.value_counts().values
test_counts = y_test.value_counts().values
chi2, p_val = stats.chisquare(test_counts, f_exp=train_counts * len(y_test) / len(y_train))

print(f"\nChi-squared test: χ²={chi2:.3f}, p={p_val:.4f}")
if p_val > 0.05:
    print("✓ Class distributions are similar (p > 0.05)")
else:
    print("⚠ Class distributions differ significantly")

### 3.2 Feature Distributions

Compare feature distributions between train and test using Kolmogorov-Smirnov test.

In [ ]:
# KS test for each feature
ks_results = []

for col in X_train.columns:
    ks_stat, p_val = stats.ks_2samp(X_train[col], X_test[col])
    ks_results.append({
        'feature': col,
        'ks_statistic': ks_stat,
        'p_value': p_val,
        'similar': p_val > 0.05
    })

ks_df = pd.DataFrame(ks_results).sort_values('p_value')

n_similar = ks_df['similar'].sum()
print(f"KS test results: {n_similar}/{len(X_train.columns)} features have similar distributions (p > 0.05)")

if n_similar == len(X_train.columns):
    print("✓ All feature distributions preserved across splits")
elif n_similar / len(X_train.columns) > 0.9:
    print("✓ Most feature distributions preserved (>90%)")
else:
    print("⚠ Some feature distributions differ between train/test")
    print("\nFeatures with different distributions:")
    display(ks_df[~ks_df['similar']][['feature', 'p_value']])

## 4. Transformation Validation

Verify transformations were applied correctly.

In [ ]:
# Check standardization (mean ≈ 0, std ≈ 1 for train)
print("Standardization check (train set):")
train_means = X_train.mean()
train_stds = X_train.std()

# Find numeric columns
numeric_cols = X_train.select_dtypes(include=[np.number]).columns

# Check if standardized
well_standardized = (
    (abs(train_means[numeric_cols]) < 0.1).all() and 
    (abs(train_stds[numeric_cols] - 1.0) < 0.1).all()
)

if well_standardized:
    print("✓ Features are well-standardized (mean ≈ 0, std ≈ 1)")
else:
    print("Note: Some features may not be standardized (e.g., binary encoded)")

print(f"\nMean range: [{train_means.min():.3f}, {train_means.max():.3f}]")
print(f"Std range: [{train_stds.min():.3f}, {train_stds.max():.3f}]")

## 5. Generate Validation Report

In [ ]:
# Validation report
validation_report = {
    'notebook': '02d_preprocessing_validation.ipynb',
    'data_leakage_check': {
        'overlapping_indices': len(overlap),
        'passed': len(overlap) == 0
    },
    'class_distribution': {
        'train_positive_rate': float(y_train.mean()),
        'test_positive_rate': float(y_test.mean()),
        'chi2_p_value': float(p_val),
        'passed': p_val > 0.05
    },
    'feature_distributions': {
        'n_features': len(X_train.columns),
        'n_similar': int(n_similar),
        'pct_similar': float(n_similar / len(X_train.columns)),
        'passed': n_similar / len(X_train.columns) > 0.9
    },
    'transformations': {
        'standardized': well_standardized,
        'train_mean_range': [float(train_means.min()), float(train_means.max())],
        'train_std_range': [float(train_stds.min()), float(train_stds.max())]
    },
    'overall_validation': 'PASSED' if (len(overlap) == 0 and p_val > 0.05 and n_similar / len(X_train.columns) > 0.9) else 'CHECK_ISSUES'
}

# Save report
with open(METADATA_DIR / "preprocessing_validation_report.json", 'w') as f:
    json.dump(validation_report, f, indent=2)

print("\n" + "="*60)
print("PREPROCESSING VALIDATION REPORT")
print("="*60)
print(f"\nData Leakage: {'✓ PASS' if validation_report['data_leakage_check']['passed'] else '✗ FAIL'}")
print(f"Class Distribution: {'✓ PASS' if validation_report['class_distribution']['passed'] else '✗ FAIL'}")
print(f"Feature Distributions: {'✓ PASS' if validation_report['feature_distributions']['passed'] else '✗ FAIL'}")
print(f"\nOverall: {validation_report['overall_validation']}")
print("\n✓ Saved: preprocessing_validation_report.json")
print("\nNext: 03a_logistic_regression.ipynb")